# Qualtran-L1 Intermediate Representation

The L1 intermediate representation (IR) for Qualtran is a host-language-agnostic interchange format for hierarchical composite quantum programs. The Python `qualtran` library (Qualtran-L2) can compile to Qualtran-L1.

 - L1 quantum functions are represented as text vs. L2 bloqs represented as in-memory Python objects.
 - a L1 `*.qlt` file contains a full hierarchical record of the quantum funciton vs. "lazy" evaluation of decompositions in L2.
 - Compile-time classical parameters are filled in and static in L1 vs. templated families of quantum subroutines via bloq classes in L2.

## Example usage

In [ ]:
import qualtran as qlt
import qualtran.dtype as qdt

### `dump_l1`

This function will take a bloq and recursively "compile" it to the static L1 textual representation. 

 - The hierarchical structure of the program definition is maintained.
 - Each bloq object is included as a `qdef` quantum function declaration with a static quantum signature (enclosed in square brackets).
 - Bloq objects with a defined decomposition include a `qdef` quantum function definition with a static function body (enclosed in curly braces) encoding the composition.
 - Bloq objects without a defined decomposition are included as `extern qdef` declarations *only*. These must include the `from` clause, which provides the L1-objectstrng used to *link* the extern qdefs with an in-memory Python bloq object.

In [ ]:
from qualtran.bloqs.arithmetic import BitwiseNot
from qualtran.l1 import dump_l1

l1_code = dump_l1(BitwiseNot(qdt.QUInt(4)))
print(l1_code)

### `dump_root_l1`

This convenience function is provided to only dump one level of decomposition and declare each of the immediate callees as extern qdefs.

In [ ]:
from qualtran.bloqs.arithmetic import Negate
from qualtran.l1 import dump_root_l1

negate = Negate(qdt.QUInt(8))
root_l1_code = dump_root_l1(negate)
print(root_l1_code)

### `load_module`

In [ ]:
from qualtran.l1 import load_module
module = load_module("""
# Qualtran-L1
# 1.0.0

qdef Negate[x: QUInt(8)] {
    x2 = BitwiseNot(8)[x=x]               
    x3 = AddK(k=1)[x=x2]              
    return [x=x3]              
}

extern qdef BitwiseNot(8)
from qualtran.bloqs.arithmetic.BitwiseNot(QUInt(8))
[x: QUInt(8)]

extern qdef AddK(k=1)
from qualtran.bloqs.arithmetic.AddK(QUInt(8), 1)
[x: QUInt(8)]""")
module.keys()

In [ ]:
module['Negate'].bloq_instances

In [ ]:
type(module['Negate']), type(negate)

In [ ]:
module['Negate'].call_classically(x=1)